In [14]:
import ansys.fluent.core as pyfluent
import ansys.fluent.core.parametric as para
from ansys.fluent.visualization import (
    Contour,
    GraphicsWindow,
    Mesh,
    Monitor,
    Pathline,
    Surface,
    Vector,
    XYPlot,
    config,
)
%matplotlib notebook
import numpy as np
import os
import math as math
import time as time
import warnings

In [15]:
working_directory = os.getcwd()

In [16]:
from general_setting_setup import steady_settings
from general_setting_setup import general_settings_transients
from create_graphics import create_particle_track
from peak_velocity_calc import find_peak_velo
from assign_named_expressions import assign_named_expressions

In [17]:
#calculations for inhalation rate and velocity
respiratory_rate = 13.2 #bpm
inhalation_time = 20/respiratory_rate #sec
peak_inhalation_rate = 27*respiratory_rate*np.pi/40 #L/min

In [ ]:
size_of_timestep = 0.0001
seconds_to_simulate = 0.1 -2*size_of_timestep
number_of_timesteps = math.ceil(seconds_to_simulate/size_of_timestep)
iterations_per_timestep = 50
path_to_cases_and_datas = "D:\\Lii\\Results\\New-tests\\independence-test\\case-and-data\\"
#these case files have run for 2 time steps
case_file_names = ["235Mcells_test_3.cas.h5",
                   "168Mcells_test_2.cas.h5",
                   "389Mcells_test_4.cas.h5",
                   "481Mcells_test_5.cas.h5",
                   "633Mcells_test_6.cas.h5",
                   "888kcells_test_1.cas.h5",
                   "935Mcells_test_1.cas.h5"]
mesh_file_names = ["new-model-only-decimated-1-06skew-035orth-235Mcells.msh.h5",
                   "new-model-only-decimated-1-06skew-032orth-168Mcells.msh.h5",
                   "new-model-only-decimated-1-06skew-031orth-389Mcells.msh.h5",
                   "new-model-only-decimated-1-06skew-032orth-481Mcells.msh.h5",
                   "new-model-only-decimated-1-06skew-033orth-633Mcells.msh.h5",
                   "new-model-only-decimated-1-07skew-025orth-888kcells.msh.h5",
                   "new-model-only-decimated-1-06skew-032orth-935Mcells.msh.h5"]

In [19]:
#launch fluent in solver
solver = pyfluent.launch_fluent(precision='double', 
                                processor_count=12, 
                                ui_mode='gui', 
                                mode='solver', 
                                start_transcript=False)
warnings.filterwarnings("ignore")

In [20]:
#calculate peak inhalation velocity
solver.settings.file.read_mesh(file_name=str("Meshes\\test-new-model-only-decimated-1-07skew-025orth-903kcells.msh.h5"))
peak_inhalation_velocity = find_peak_velo(solver_session = solver,
                                          working_directory=working_directory,
                                          peak_flow_rate = peak_inhalation_rate,
                                          calc_area = "nostrils",
                                          area_file_name = "surface_integral",
                                          unit = "L/min")

In [22]:
i=0
for i in range(len(mesh_file_names)):

#create file name part
    name = str(case_file_names[i][0:case_file_names[i].find(".cas")]) # should now be [nr of cells]_test_[number]

#read the transient result case and data file
    solver.settings.file.read_case(file_name=str(path_to_cases_and_datas+case_file_names[i]))

#read the mesh file
    solver.settings.file.replace_mesh(file_name = "Meshes\\" + mesh_file_names[i])

# read data file
    solver.settings.file.read_data(file_name=str(path_to_cases_and_datas+name+".dat.h5"))

# CONTINUE TRANSIENT SIMULATION

#start dpm sample
    solver.settings.file.read_journal(file_name_list=["start-transient-dpm-sample.jou"])

# set calculation settings for transient part and run the calculation
    controls = solver.settings.solution.run_calculation.transient_controls
    controls.time_step_count = number_of_timesteps
    controls.time_step_size = size_of_timestep
    controls.max_iter_per_time_step = iterations_per_timestep
    solver.settings.solution.run_calculation.calculate()

#end dpm sample
    time.sleep(3)
    solver.settings.file.read_journal(file_name_list=["stop-transient-dpm-sample.jou"])

#Save the new transient results
    case_data_name = str("D:\\Lii\\Results\\New-tests\\transient-independence-test\\" + str(name) + "_transient")
    solver.tui.file.write_case_data(case_data_name)
# Rename dpm files
    os.rename("interior--facetten.dpm", "interior--facetten_" + name + ".dpm")
    os.rename("new-model-only-decimated-1-facetten.dpm", "new-model-only-decimated-1-facetten_" + name + ".dpm")


KeyboardInterrupt: 

In [23]:
solver.exit()